## **_Sample Modeling_** (for Reinforcement Learning)

이 페이지는 "Drying Oven"의 가상 센서 데이터를 이용하여 PPO 모델을 교육하고 예측하는 예제를 구현합니다. 

#### Step 1 가상 센서 데이터 생성

**시나리오**
문 열림이나 외부 간섭으로 인해 내부 온도가 급격히 떨어질 때, 공기의 흐름(풍압)도 함께 불규칙하게 요동치는 상황을 반영했습니다. 보통 문이 열리면 외부 공기 유입으로 인해 압력이 순간적으로 변하고, 시스템은 이를 보정하기 위해 팬 RPM을 급격히 조절하게 됩니다.

**주요 포인트**

* Drop & Recovery: drop_start 지점에서 온도가 약 30~40도가량 빠르게 떨어졌다가 다시 heating 모드를 통해 target_temp로 복귀합니다.
* 데이터 비중: 하락 및 복구 구간(약 110개 데이터)을 제외하고도 약 1,500개 이상의 데이터가 stable 상태를 유지하여 요청하신 80% 비중을 최대한 맞췄습니다.
* 학습 효과: 강화학습 모델이 이 데이터를 학습하면 "온도가 급격히 변할 때 히터 상태를 어떻게 바꿔야 하는가"에 대한 대응책을 배울 수 있습니다.

**추가된 로직 설명**

* 풍압 요동(Pressure Fluctuation): drop_start 지점에서 풍압이 ±10.0 범위로 크게 튀게 설정했습니다. 이는 실제 환경에서 문이 열릴 때 발생하는 압력 차를 모사한 것입니다.
* 팬 RPM 연동: 풍압이 요동치면 시스템이 이를 보정하려고 하므로, RPM 값도 풍압에 따라 급격히 변화하도록 수식을 연결했습니다.
* 복구 패턴: 온도가 떨어지는 초기에 풍압이 가장 심하게 요동치고, 온도를 다시 올리는 복구 구간에서는 풍압이 서서히 안정화되는 흐름을 가집니다.
* 이 데이터를 사용하여 학습시킬 때, 온도 오차뿐만 아니라 풍압의 급격한 변화에 대해서도 마이너스 보상(Penalty)을 주면 더 안정적인 제어 로직을 만들 수 있습니다.

In [6]:
import pandas as pd
import numpy as np

# 1. 설정
n_samples = 2000
target_temp = 80.0
target_pressure = 15.0
data = []

# 초기값 설정
curr_temp = 25.0
curr_hum = 60.0
curr_weight = 5000.0

# 구간 설정
warm_up_end = int(n_samples * 0.1)      # 10%: 초기 가열
drop_start = int(n_samples * 0.5)       # 50% 지점: 이상 발생
recovery_end = drop_start + 100         # 100스텝: 복구 구간
stable_end = int(n_samples * 0.9)       # 이후 90%까지 안정 (전체 약 80% 비중)

for i in range(n_samples):
    # 기본 풍압 상태 (평상시 노이즈)
    press_noise = np.random.normal(0, 0.3)
    
    # 2. 구간별 제어 및 환경 변화
    if i < warm_up_end:
        h_state = 'heating'
        curr_temp = min(target_temp, curr_temp + np.random.uniform(1.5, 2.5))
        air_pressure = target_pressure + press_noise
        
    elif drop_start <= i < drop_start + 15: 
        # [급격한 하락 및 풍압 요동] - 문 열림/외부 간섭 발생
        h_state = 'stable' 
        curr_temp -= np.random.uniform(4.0, 6.0) # 온도 급락
        # 풍압이 위아래로 크게 요동 (외풍 유입 모사)
        air_pressure = target_pressure + np.random.uniform(-10.0, 10.0) 
        
    elif drop_start + 15 <= i < recovery_end:
        # [복구 구간] - 다시 목표치를 향해 제어
        h_state = 'heating'
        curr_temp = min(target_temp, curr_temp + np.random.uniform(2.0, 3.5))
        # 풍압을 다시 잡기 위해 과하게 작동하는 상황
        air_pressure = target_pressure + np.random.uniform(-2.0, 4.0)
        
    elif i < stable_end:
        # [안정 구간 - 약 80% 비중]
        h_state = 'stable'
        curr_temp = target_temp + np.random.uniform(-0.5, 0.5)
        air_pressure = target_pressure + press_noise
        
    else:
        # [마지막 변동 구간]
        h_state = 'overheat'
        curr_temp += np.random.uniform(0.2, 1.0)
        air_pressure = target_pressure + np.random.uniform(-1.0, 1.0)

    # 3. 물리적 상관관계 반영
    # RPM은 풍압을 만들기 위한 제어 결과값 (P ∝ RPM^2 역산)
    curr_rpm = np.sqrt(max(0.1, air_pressure) / 5.0) * 1000 + np.random.normal(0, 15)
    
    # 나머지 센서값
    curr_surf_temp = curr_temp - np.random.uniform(1.0, 3.0)
    curr_hum = max(10.0, 60.0 - (curr_temp - 25.0) * 0.8 + np.random.normal(0, 1))
    curr_weight -= np.random.uniform(0.1, 0.4)

    data.append([
        round(curr_temp, 2), round(curr_hum, 2), round(curr_weight, 2),
        round(curr_surf_temp, 2), round(curr_rpm, 1), round(air_pressure, 3),
        h_state
    ])

# 4. 데이터프레임 생성
df = pd.DataFrame(data, columns=['internal_temp', 'humidity', 'weight', 'surface_temp', 'fan_rpm', 'air_pressure', 'heater_state'])

# 결과 확인 (요동치는 구간 출력)
print("--- 이상 발생 및 복구 구간 데이터 (500~520번) ---")
print(df.iloc[drop_start-2:drop_start+18][['internal_temp', 'air_pressure', 'fan_rpm', 'heater_state']])




--- 이상 발생 및 복구 구간 데이터 (500~520번) ---
      internal_temp  air_pressure  fan_rpm heater_state
998           79.54        15.207   1761.5       stable
999           80.01        15.344   1753.2       stable
1000          74.07         7.578   1253.2       stable
1001          69.27        15.429   1746.0       stable
1002          65.17        17.267   1842.6       stable
1003          60.74        15.859   1765.0       stable
1004          55.13        12.179   1536.7       stable
1005          50.10        16.339   1831.3       stable
1006          44.98        20.744   2037.9       stable
1007          39.42         6.775   1156.3       stable
1008          34.53        20.086   2027.1       stable
1009          30.51        15.916   1775.8       stable
1010          24.96        13.953   1648.6       stable
1011          20.51        14.872   1723.4       stable
1012          16.38        10.427   1448.1       stable
1013          11.45        14.263   1693.1       stable
1014       

#### Step 2 Env 생성

##### Step 2.1 보상 코드

데이터에 급격한 온도 하락과 풍압 요동 시나리오가 포함되었으므로, 강화학습 에이전트가 이러한 비정상 상황(Abnormal State)을 빠르게 감지하고 복구하도록 유도하는 보상 함수(Reward Function)를 설계했습니다.

단순히 현재 온도만 보는 것이 아니라, 풍압의 안정성과 제어의 일관성을 모두 고려한 코드입니다.

1. 강화학습 환경(Env) 내 보상 함수 구현

- 이 보상 함수는 에이전트가 다음 세 가지 목표를 동시에 달성하도록 유도합니다.
- 온도 유지: 목표 온도(80도)와의 차이 최소화
- 풍압 안정: 목표 풍압(15Pa)과의 차이 최소화 및 요동 방지
- 제어 효율: 불필요하게 팬 RPM이나 히터를 과하게 조작하지 않음 (Penalty)

In [ ]:
def calculate_reward(state, action, prev_action, target_temp=80.0, target_press=15.0):
    # state: [internal_temp, humidity, weight, surface_temp, fan_rpm, air_pressure]
    curr_temp = state[0]
    curr_press = state[5]
    
    # 1. 온도 보상 (가장 높은 가중치)
    temp_error = abs(target_temp - curr_temp)
    # 오차가 커질수록 벌점이 기하급수적으로 커짐 (Quadratic Penalty)
    r_temp = -(temp_error ** 2) 
    
    # 2. 풍압 보상
    press_error = abs(target_press - curr_press)
    r_press = -press_error * 2.0 # 풍압 오차에 대한 벌점
    
    # 3. 제어 안정성 보상 (Action Smoothing)
    # 이전 액션과 현재 액션의 차이가 크면 벌점 (기계의 수명 보호 및 급격한 요동 방지)
    control_delta = np.sum(np.abs(action - prev_action))
    r_control = -control_delta * 0.5
    
    # 4. 가중치 합산 (온도 7 : 풍압 2 : 안정성 1)
    total_reward = (0.7 * r_temp) + (0.2 * r_press) + (0.1 * r_control)
    
    # 추가: 온도가 위험 범위(120도 이상)를 넘어가면 매우 큰 감점
    if curr_temp > 120:
        total_reward -= 100
        
    return total_reward


##### 2.2 학습 파이프라인 코드

생성된 가상 센서 데이터를 불러와서 강화학습 모델(PPO)이 환경의 특성을 학습하고, 최적의 제어 정책을 도출하는 전체 학습 파이프라인(Pipeline) 코드입니다.

이 코드는 Gymnasium 환경 내에서 앞서 만든 가상 데이터의 물리 법칙(온도 하락, 풍압 요동 등)을 시뮬레이션하며 에이전트를 학습시킵니다.

**_파이프라인의 핵심 구성 요소_**

* Gymnasium Env Interface: step() 함수 내에 에이전트의 액션이 실제 온도와 풍압에 어떻게 영향을 주는지 물리 법칙을 정의했습니다.
* PPO 알고리즘: Stable Baselines3의 PPO는 제어 시스템에서 안정적인 성능을 보여주는 대표적인 강화학습 알고리즘입니다.
* Quadratic Penalty: 온도 오차를 제곱(temp_err**2)하여 벌점을 부여함으로써, 목표 온도에서 멀어질수록 에이전트가 더 강하게 반응하도록 유도했습니다.
* Control Smoothing: ctrl_penalty를 통해 히터나 팬을 너무 급격하게 조작하지 않도록 제한하여 실제 기계의 마모를 고려했습니다.
이 파이프라인을 실행하면 에이전트가 초기 25도에서 시작하여 80도 온도와 15Pa 풍압을 찾아가고 유지하는 법을 스스로 터득하게 됩니다.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
import matplotlib.pyplot as plt

# 1. 강화학습 환경 정의 (Drying Oven Environment)
class DryingOvenEnv0(gym.Env):
    def __init__(self):
        super(DryingOvenEnv, self).__init__()
        
        # Action: [히터 출력 증감(-1~1), 팬 RPM 증감(-1~1)]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)
        
        # Observation: [내부온도, 습도, 무게, 표면온도, 현재RPM, 풍압]
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(6,), dtype=np.float32)
        
        self.target_temp = 80.0
        self.target_press = 15.0
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # 초기 상태 설정
        self.state = np.array([25.0, 60.0, 5000.0, 24.0, 1500.0, 15.0], dtype=np.float32)
        self.prev_action = np.zeros(2)
        self.steps = 0
        return self.state, {}

    def step(self, action):
        curr_temp, hum, weight, surf_temp, rpm, press = self.state
        h_ctrl, r_ctrl = action # 에이전트의 제어 신호
        
        # --- 물리 엔진 모사 (생성 데이터의 로직 반영) ---
        # 1. 온도 변화: 히터 제어 + 풍압에 의한 냉각 효과
        new_temp = curr_temp + (h_ctrl * 3.0) - (press * 0.1) + np.random.normal(0, 0.1)
        
        # 2. 풍압 변화: RPM 제어 반영 (P ∝ RPM^2)
        new_rpm = np.clip(rpm + (r_ctrl * 150), 500, 3500)
        new_press = (new_rpm / 1000)**2 * 5.0 + np.random.normal(0, 0.5)
        
        # 3. 기타 변수 (습도, 무게, 표면온도 연동)
        new_hum = max(10.0, hum - (h_ctrl * 0.2))
        new_weight = weight - 0.2
        new_surf_temp = new_temp * 0.95
        
        self.state = np.array([new_temp, new_hum, new_weight, new_surf_temp, new_rpm, new_press], dtype=np.float32)
        
        # --- Reward 설계 (앞서 정의한 로직) ---
        temp_err = abs(self.target_temp - new_temp)
        press_err = abs(self.target_press - new_press)
        ctrl_penalty = np.sum(np.abs(action - self.prev_action))
        
        # 보상 가중치: 온도(0.7), 풍압(0.2), 제어안정성(0.1)
        reward = -(0.7 * (temp_err**2) + 0.2 * press_err + 0.1 * ctrl_penalty)
        
        # 이상 상황 종료 조건 (안전 장치)
        terminated = bool(new_temp > 120 or new_temp < 10)
        self.prev_action = action
        self.steps += 1
        truncated = self.steps >= 500 # 한 에피소드당 500스텝 제한
        
        return self.state, reward, terminated, truncated, {}

##### Step 2.2.3 수정된 보상 함수 및 파이프라인 (풍압 중심 제어)

보상 함수에서 풍압(air_pressure)의 비중을 대폭 높이고, 에이전트가 풍압을 목표치(15Pa)에 맞추기 위해 팬 RPM을 정밀하게 제어하도록 로직을 수정했습니다.

이제 온도는 기본적인 제어 대상이 되며, 풍압의 안정성이 보상의 핵심 지표가 됩니다.

보상 로직의 특징
- Quadratic Pressure Penalty: 풍압 오차에 제곱(\^2)과 가중치(5.0)를 적용하여, 에이전트가 풍압을 15Pa에 맞추는 것을 최우선 순위로 학습하게 했습니다.
- Pressure Delta Reward: 풍압이 목표치로 접근하는 '방향성'에 보상을 주어, 급격한 요동 구간에서도 에이전트가 포기하지 않고 제어력을 유지하도록 돕습니다.
- Complex Interaction: 풍압이 높아지면 온도가 낮아지는 냉각 효과가 반영되어 있으므로, 에이전트는 온도를 지키면서도 풍압을 최적화하는 미세한 밸런스를 찾게 됩니다.


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
import matplotlib.pyplot as plt

class DryingOvenEnv(gym.Env):
    def __init__(self):
        super(DryingOvenEnv, self).__init__()
        # 제어: [히터 출력 증감, 팬 RPM 증감]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)
        # 관측: [내부온도, 습도, 무게, 표면온도, 현재RPM, 풍압]
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(6,), dtype=np.float32)
        
        self.target_temp = 80.0
        self.target_press = 15.0 # 보상의 핵심 목표
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.state = np.array([25.0, 60.0, 5000.0, 24.0, 1500.0, 15.0], dtype=np.float32)
        self.prev_press_error = 0.0
        self.steps = 0
        return self.state, {}

    def step(self, action):
        curr_temp, hum, weight, surf_temp, rpm, press = self.state
        h_ctrl, r_ctrl = action
        
        # 1. 물리 시뮬레이션 (RPM이 풍압에 직접 영향)
        new_rpm = np.clip(rpm + (r_ctrl * 200), 500, 3500)
        # 풍압 물리 법칙: P = (RPM/1000)^2 * 5 + 외부 요동(noise)
        new_press = (new_rpm / 1000)**2 * 5.0 + np.random.normal(0, 0.2)
        
        # 온도는 히터와 풍압(냉각효과)의 영향을 받음
        new_temp = curr_temp + (h_ctrl * 4.0) - (new_press * 0.15)
        
        self.state = np.array([new_temp, hum-0.1, weight-0.1, new_temp*0.9, new_rpm, new_press], dtype=np.float32)
        
        # --- 핵심 수정: 풍압 중심의 보상 설계 (Reward Design) ---
        press_error = abs(self.target_press - new_press)
        temp_error = abs(self.target_temp - new_temp)
        
        # 1) 풍압 보상 (매우 강력한 페널티): 목표 풍압에서 멀어질수록 벌점 증가
        r_press = -(press_error ** 2) * 5.0 
        
        # 2) 풍압 안정성 보상: 이전 스텝보다 풍압 오차가 줄어들면 추가 보상
        r_press_delta = (self.prev_press_error - press_error) * 2.0
        
        # 3) 온도 보상: 기본 수준 유지
        r_temp = -(temp_error ** 2) * 0.5
        
        # 최종 보상 합산 (풍압 가중치 극대화)
        reward = r_press + r_press_delta + r_temp
        
        self.prev_press_error = press_error
        self.steps += 1
        
        terminated = bool(new_temp > 130 or new_press > 50)
        truncated = self.steps >= 500
        
        return self.state, reward, terminated, truncated, {}

# 학습 및 결과 확인
env = DryingOvenEnv()
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=15000)

# 테스트 실행
obs, _ = env.reset()
press_history = []
for _ in range(100):
    action, _ = model.predict(obs)
    obs, r, _, _, _ = env.step(action)
    press_history.append(obs[5]) # air_pressure 기록

plt.plot(press_history, label='Air Pressure (Pa)')
plt.axhline(15, color='r', linestyle='--', label='Target')
plt.title("Reward-Driven Pressure Control")
plt.legend()
plt.show()


#### Step 3 모방 학습(Behavioral Cloning)으로 PPO 초기화하기

##### Step 3.1 가상 데이터를 Transitions 데이터로 변환

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

df = pd.read_csv(csv_file_name)
# df = pd.DataFrame(data, columns=['internal_temp', 'humidity', 'weight', 'surface_temp', 'fan_rpm', 'air_pressure', 'heater_state'])

# obs = df[['internal_temp', 'humidity', 'weight', 'surface_temp', 'fan_rpm', 'air_pressure']].values.astype(np.float32)
# acts = df[['heater_state', 'fan_rpm']].values.astype(np.float32)
obs = df[['internal_temp', 'humidity', 'weight', 'surface_temp', 'fan_rpm', 'air_pressure']]
acts = df[['heater_state', 'fan_rpm']]
obs = torch.tensor(obs.values, dtype=torch.float32)
acts = torch.tensor(acts.values, dtype=torch.float32)
print(len(obs))
print(len(acts))
dataset = TensorDataset(obs, acts)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# next_obs는 보통 현재 obs의 한 칸 아래(t+1) 행 데이터입니다.
next_obs = np.roll(obs, -1, axis=0) 
# 마지막 행은 다음 데이터가 없으므로 제외하거나 처리 필요
transitions = types.Transitions(obs=obs[:-1], acts=acts[:-1], next_obs=next_obs[:-1], 
                                dones=dones[:-1], infos=[{}]*(len(df)-1))



##### 3.2 Trasitions 데이터로 모방 학습 하기

* PyTorch를 이용
* 가상 데이터 준비 및 BC 학습 (Pre-training)
* 가상 데이터에서 (State) -> (Action) 관계를 지도 학습(Supervised Learning)으로 먼저 익힙니다.

3.2.1 imitatin을 이용한 방법

In [ ]:
from imitation.algorithms import bc
from stable_baselines3 import PPO

bc_trainer = bc.BC(
    observation_space=env.observation_space,
    action_space=env.action_space,
    demonstrations=transitions, # 가상 데이터 입력
)
bc_trainer.train(n_epochs=50)

3.2.2 PyTorch를 이용한 방법

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from stable_baselines3 import PPO

# BC 모델 정의 (PPO의 Policy 구조와 맞춤)
class BC_Policy(nn.Module):
    def __init__(self):
        super(BC_Policy, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(6, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2)
        )
    def forward(self, x): return self.net(x)

bc_model_torch = BC_Policy()
optimizer = optim.Adam(bc_model_torch.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# BC 학습 실행
print("Behavioral Cloning 학습 중...")
for epoch in range(50):
    for s, a in loader:
        optimizer.zero_grad()
        loss = criterion(bc_model_torch(s), a)
        loss.backward()
        optimizer.step()
print(f"BC 완료. 최종 Loss: {loss.item():.4f}")


I0000 00:00:1774097404.151223  910737 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774097404.987813  910737 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774097409.361810  910737 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/tmp/ipykernel_910737/254286874.py:10: UserWarning: The given NumPy array is not writable, and PyTorch do

Behavioral Cloning 학습 중...
BC 완료. 최종 Loss: 0.2741


#### 4 PPO 모델 생성

4.1 imitation을 이용하여 생성 된 policy 이용

In [ ]:
env = DryingOvenEnv() # 앞서 정의한 풍압 중심 Env
ppo_model = PPO("MlpPolicy", env, verbose=1)
ppo_model.policy = bc_trainer.policy # BC 가중치를 PPO로 복사

# 4. PPO 재학습 (강화학습 시작)
ppo_model.learn(total_timesteps=20000)

# 학습 결과 저장
ppo_model.save("oven_control_model")

4.2 Pytorch를 이용하여 생성 된 policy 이용

In [ ]:
env = DryingOvenEnv() # 앞서 정의한 풍압 중심 Env
# 모델 설정 (PPO 알고리즘 사용)
# MlpPolicy: 센서 데이터(벡터) 처리에 적합한 다층 퍼셉트론 신경망
ppo_model_torch = PPO("MlpPolicy", env, verbose=1, learning_rate=3e-4, device='cpu')

# BC 모델의 가중치를 PPO의 actor 네트워크로 복사
# SB3의 policy 구조에 맞춰 매핑 (간략화된 예시)
with torch.no_grad():
    ppo_model_torch.policy.action_net.weight.copy_(bc_model_torch.net[-1].weight)
    ppo_model_torch.policy.action_net.bias.copy_(bc_model_torch.net[-1].bias)

# 5. 강화학습으로 Fine-tuning (전이 학습)
print("BC 정책을 기반으로 RL Fine-tuning 시작...")
ppo_model_torch.learn(total_timesteps=200000) # 2만 번의 시행착오를 통해 학습

# 6. 결과 저장
ppo_model_torch.save("bc_ppo_oven_model")
print("모델 저장 완료: oven_control_model.zip")

##### 5. Test

In [ ]:
# 테스트 실행
use_torch = True

model = ppo_model_torch if use_torch == True else ppo_model

obs, _ = env.reset()
press_history = []

for _ in range(100):
    action, _ = model.predict(obs)
    obs, r, _, _, _ = env.step(action)
    press_history.append(obs[5]) # air_pressure 기록

plt.plot(press_history, label='Air Pressure (Pa)')
plt.axhline(15, color='r', linestyle='--', label='Target')
plt.title("Reward-Driven Pressure Control")
plt.legend()
plt.show()

#### 6. 센서 데이터 읽어서 예측 결과를 모터에 전송하기

** 센서 데이터 읽기 **

* 실제 환경에 적용하기 위해 외부 센서 데이터(CSV, API, 또는 PLC 통신 등)를 실시간으로 읽어와서 학습된 BC-PPO 모델에 입력하고, 제어 명령(Action)을 출력하는 루프를 추가했습니다.
* 센서 데이터를 읽어오는 부분은 실제 환경에 맞춰 교체할 수 있도록 read_external_sensors() 함수로 추상화했습니다.
* 주요 연동 포인트
  * Observation 정규화: 학습 때 사용한 데이터의 범위와 실제 센서 데이터의 범위가 크게 다를 경우, (obs - mean) / std와 같은 Scaling 과정을 read_external_sensors 내에 추가해야 모델이 정확하게 판단합니다.
  * Deterministic Prediction: 실전 제어에서는 확률적인 탐색보다는 모델이 가장 좋다고 판단하는 확정적인 값(deterministic=True)을 사용하는 것이 안전합니다.
  * Action Mapping: 모델의 출력값은 보통 -1 ~ 1 사이의 수치입니다. 이를 실제 히터의 전압(V), 전류(mA), 혹은 PWM % 값으로 변환하는 매핑 로직이 send_control_signal에 필요합니다.

** 모터 제어 **

* 학습된 BC-PPO 모델의 예측값(Action)을 받아 실제 모터의 RPM 제어 신호로 변환하고, 이를 장비(모터 드라이버 등)에 전달하는 과정을 포함한 최종 제어 루프입니다.
* 보통 산업용 모터는 Modbus TCP나 0~10V 아날로그 신호 등을 사용하므로, 이를 모사한 변환 로직을 apply_motor_control 함수에 추가했습니다.
* 코드 핵심 포인트
  * RPM Delta 제어: 모델의 출력을 절대적 RPM 값이 아닌 변화량(rpm_delta)으로 사용했습니다. 이는 모터에 급격한 부하가 걸리는 것을 방지하고 부드러운 가속/감속을 가능하게 합니다.
  * Hard Clipping: np.clip을 사용하여 모델이 실수로 장비의 물리적 한계(500~3000 RPM)를 넘는 명령을 내려도 안전하게 차단합니다.
Global State 유지: CURRENT_MOTOR_RPM을 추적하여 현재 상태를 기준으로 다음 제어량을 결정하도록 설계했습니다.

In [8]:
import time
import numpy as np
from stable_baselines3 import PPO

# 1. 학습된 모델 로드
model = PPO.load("bc_ppo_oven_model")

# [설정] 모터 사양 및 제어 범위
MIN_RPM = 500
MAX_RPM = 3000
CURRENT_MOTOR_RPM = 1500  # 초기 가동 RPM

def read_external_sensors():
    """
    외부 센서로부터 현재 상태를 읽어옴 (예시 데이터)
    순서: [internal_temp, humidity, weight, surface_temp, fan_rpm, air_pressure]
    """
    # 실제 환경에서는 PLC/센서 API 호출 결과가 들어감
    return np.array([79.2, 30.1, 4900.5, 76.0, CURRENT_MOTOR_RPM, 14.5], dtype=np.float32)

def apply_motor_control(fan_action):
    """
    모델의 Action(-1 ~ 1)을 실제 모터 RPM 값으로 변환하여 전송
    """
    global CURRENT_MOTOR_RPM
    
    # 1. Action(-1 ~ 1)을 RPM 변화량으로 매핑 (예: 한 번에 최대 200 RPM 증감)
    rpm_delta = fan_action * 200
    
    # 2. 새로운 목표 RPM 계산 및 하드웨어 제한(Limit) 적용
    target_rpm = np.clip(CURRENT_MOTOR_RPM + rpm_delta, MIN_RPM, MAX_RPM)
    
    # 3. 실제 모터 드라이버에 명령 전달 (예: Modbus Write 또는 DAC 출력)
    # write_to_motor_driver(target_rpm) # 실제 통신 함수 가정
    
    CURRENT_MOTOR_RPM = target_rpm # 현재 상태 업데이트
    return target_rpm

def apply_heater_control(heater_action):
    """히터 제어 신호 변환 (예: 0~100% PWM)"""
    heater_power = np.clip((heater_action + 1) * 50, 0, 100)
    # write_to_heater_relay(heater_power)
    return heater_power

# 2. 실시간 제어 루프
print("--- Drying Oven AI-Driven Motor & Heater Control Start ---")

try:
    while True:
        # (1) 센서 데이터 수집
        obs = read_external_sensors()
        
        # (2) 모델 추론 (Deterministic=True로 안정성 확보)
        # obs shape: (6,) -> (1, 6)으로 모델 입력
        action, _ = model.predict(obs, deterministic=True)
        
        # action[0]: 히터 제어, action[1]: 팬(풍압) 제어
        h_action, f_action = action[0], action[1]
        
        # (3) 하드웨어 제어 명령 실행
        target_rpm = apply_motor_control(f_action)
        heater_pwr = apply_heater_control(h_action)
        
        # (4) 모니터링 출력
        print(f"[상태] 온도: {obs[0]:.1f}°C | 풍압: {obs[5]:.2f}Pa")
        print(f"[제어] 목표 RPM: {target_rpm:.0f} | 히터: {heater_pwr:.1f}%")
        print("-" * 40)
        
        # 제어 주기 (예: 2초마다 갱신)
        time.sleep(2.0)

except KeyboardInterrupt:
    print("\n--- 시스템 안전 종료 및 모터 정지 ---")
    # apply_motor_control(-1) # 모터 최소화 또는 정지 로직


FileNotFoundError: [Errno 2] No such file or directory: 'bc_ppo_oven_model.zip'